In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from Utils.TrainUtils import get_device, build_dataloaders, train_model

from Models.FraudModel import TransformerFraudModel, LastTokenMLP

from Utils.NormalizationUtils import fit_feature_normalizer, apply_feature_normalizer

In [2]:
stats = fit_feature_normalizer(
    processed_dir="data/processed_fraud",
    output_stats_path="data/processed_fraud/normalization_stats.json",
)

apply_feature_normalizer(
    source_processed_dir="data/processed_fraud",
    target_processed_dir="data/processed_fraud_normalized",
    stats_path="data/processed_fraud/normalization_stats.json",
)

In [3]:
device = get_device()
print(device)

train_loader, val_loader, test_loader = build_dataloaders(
    processed_dir="data/processed_fraud_normalized",
    seq_len=8,
    batch_size=128,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

feature_dim = train_loader.dataset.feature_dim
print("Feature dim: ", feature_dim)

cuda
Feature dim:  111


In [6]:
model = TransformerFraudModel(
    feature_dim=feature_dim,
    seq_len=8,
    d_model=64,
    nhead=4,
    num_layers=3,
    dim_feedforward=128,
    dropout=0.1,
)

c:\Users\mengt\Documents\DeepLearning\DL_project\Models\FraudModel.py:114: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.sequence_encoder = nn.TransformerEncoder(


In [4]:
model = LastTokenMLP(
    feature_dim=feature_dim,
    hidden_dim=64,
    dropout=0.1,
)

In [7]:
print(type(model))

summary = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=10,
    lr=1e-3,
    weight_decay=1e-4,
    checkpoint_dir="checkpoints/TransformerV1Norm_(8-64-4-3-128)",
    monitor="val_pr_auc",
    monitor_mode="max",
    use_pos_weight=True,
    max_grad_norm=1.0,
    early_stopping_patience=5,
)

<class 'Models.FraudModel.TransformerFraudModel'>
[info] Using pos_weight=27.4009

Epoch 1/10
train_loss=0.9788  val_loss=0.9362
train_pr_auc=0.3535  val_pr_auc=0.3522
train_roc_auc=0.8234  val_roc_auc=0.8201
train_f1=0.2055  val_f1=0.1656
train_acc=0.8179  val_acc=0.7767
lr=0.001000  epoch_time=36.59s
[checkpoint] Saved new best model -> checkpoints\TransformerV1Norm_(8-64-4-3-128)\best.pt

Epoch 2/10
train_loss=0.8835  val_loss=0.9014
train_pr_auc=0.4443  val_pr_auc=0.3895
train_roc_auc=0.8607  val_roc_auc=0.8311
train_f1=0.2495  val_f1=0.1809
train_acc=0.8499  val_acc=0.8023
lr=0.001000  epoch_time=37.52s
[checkpoint] Saved new best model -> checkpoints\TransformerV1Norm_(8-64-4-3-128)\best.pt

Epoch 3/10
train_loss=0.8444  val_loss=0.8685
train_pr_auc=0.4824  val_pr_auc=0.3995
train_roc_auc=0.8739  val_roc_auc=0.8470
train_f1=0.2720  val_f1=0.2010
train_acc=0.8644  val_acc=0.8246
lr=0.001000  epoch_time=39.62s
[checkpoint] Saved new best model -> checkpoints\TransformerV1Norm_(8-64